# Train random forest classifiers for variant filtering

In previous work (all done in R) I showed that a RF, using most of the features produced by rastair2, can distinguish SNPs from REF positions in 15x sequenced TAPS data with ~99.7% accuracy, and approx 90% PPV.

I would like to perform this kind of filtering in the actual rastair code, but to do so, I need to create a serialisable version of the classifiers.

It seems to be impossible to train in R and export into a format that Rust can understand, so instead I will just write some code here to perform the training in Rust directly.

We will still perform the data pre-processing in R, because I can't be bothered to figure out how to do the one-hot encoding etc in Rust.

## 0. Install dependencies

In [2]:
:dep ndarray = { version = "^0.16.1", default-features = true }
:dep flate2 = { version = "^1.1.2" }
:dep biosphere = { git = "https://github.com/DaGaMs/biosphere.git", features = ["serde"] }
:dep chrono = { version = "^0.4.41" }
:dep serde = { version = "^1.0.219" }
:dep bincode = { version = "2.0.1", features = ["serde"] }
:dep rand = { version = "^0.9.1" }
:dep anyhow = { version = "1.0.98" }
:dep smartcore = { version = "^0.4.1" }
//:dep rust-htslib = { version = "^0.49.0", git = "https://github.com/killercup/rust-htslib.git", branch = "feature/cstr8", default-features = false, features = ["bzip2", "lzma", "libdeflate"] }
:dep csv = { version = "^1.3.1" }

## CpGs
### 1. Load training data
We will produce the training data in R since that code is already there and ready to go. See `vcf_to_train.ipynb` for details.

In [3]:
use anyhow::{bail, Result};
use std::{fs::File, error::Error, io::{BufRead, BufReader, Read}};
use csv::{Reader, ReaderBuilder};
use ndarray::prelude::*;
use ndarray::{Axis, concatenate};
use flate2::read::GzDecoder;

enum FeatureType {
    CpG,
    NewCpG,
    Other
}
use FeatureType::*;

fn _read_all_rows<R: std::io::Read>(reader: &mut Reader<R>) -> Result<(Vec<f64>, Vec<f64>)> {
    // Build the CSV reader and iterate over each record.
    let mut output_vector: Vec<f64> = Vec::with_capacity(2_000_000*54);
    let mut labels: Vec<f64> = Vec::with_capacity(2_000_000);
    let mut row: usize = 0;
    let mut ncols = 0;
    for result in reader.records() {
        row = row+1;
        // The iterator yields Result<StringRecord, Error>, so we check the
        // error here..
        let record = result?;
        let num_columns = record.len();

        if ncols != num_columns {
            eprintln!("Found row with {} columns", num_columns);
            ncols = num_columns;
        }

        let label = record.get(num_columns-1).unwrap_or("REF");
        if label == "REF" {
            labels.push(0.0);
        } else {
            labels.push(1.0);
        }

        for i in 1..(num_columns-1) {
            let v = record.get(i).unwrap_or("0.0");
            output_vector.push(v.parse().unwrap_or_default());
        }
    }
    Ok((output_vector, labels))
}

fn tsv_to_matrix(path: &str, ftype: FeatureType) -> Result<(ndarray::Array2<f64>, ndarray::Array1<f64>)> {
    let file = File::open(path)?;
    let decoder = GzDecoder::new(file);
    let reader = BufReader::new(decoder);

    let mut rdr = ReaderBuilder::new().delimiter(b'\t').from_reader(reader);
    let (data, labels) = _read_all_rows(&mut rdr)?;
    let n_observations = labels.len();
    let num_features: usize = match ftype {
        CpG => 55,
        NewCpG => 55,
        Other   => 54
    };
    println!("Finished reading {} rows into a vector of size {}", n_observations, data.len());

    let data_matrix = ndarray::Array::from_shape_vec((n_observations, num_features), data)?;
    Ok((data_matrix, ndarray::Array::from_vec(labels)))
}

let cpg_data_file = "data/chr12_CpG_features.txt.gz";
let (full_data_matrix_cpg, labels_cpg) = tsv_to_matrix(cpg_data_file, CpG).unwrap();


Found row with 57 columns


Finished reading 2206920 rows into a vector of size 121380600


Done reading training data. Now we need to subsample a set of rows and create a slice of the original matrix with only those rows for training.
### 2. Subsample training data

In [6]:
use rand::prelude::*;
use rand::seq::SliceRandom;

fn select_random_elements(
    array: &Array1<f64>,
    num_samples: usize,
    to_select: f64
) -> Result<Array1<usize>, String> {
    // Find indices of positive elements
    let valid_indices: Vec<usize> = array
        .iter()
        .enumerate()
        .filter(|x| *x.1 == to_select)
        .map(|(idx, _)| idx)
        .collect();

    if valid_indices.len() < num_samples {
        return Err(format!(
            "Not enough positive elements. Found {} positive values, but need {}",
            valid_indices.len(),
            num_samples
        ));
    }

    // Randomly sample indices
    let mut rng = rand::rng();

    let selected_indices: Vec<usize> = valid_indices
        .choose_multiple(&mut rng, num_samples)
        .cloned()
        .collect();

    Ok(Array::from_vec(selected_indices))
}

fn select_rows_by_indices(array: &Array2<f64>, indices: &Array1<usize>) -> Result<Array2<f64>> {
    // Method 1: Using ndarray's select method (most efficient)
    let selected = array.select(Axis(0), indices.as_slice().unwrap());
    Ok(selected)
}

fn combine_and_sort(arr1: Array1<usize>, arr2: Array1<usize>) -> Array1<usize> {
    // Method 1: Using ndarray's concatenate function
    let combined = concatenate![Axis(0), arr1.view(), arr2.view()];

    // Convert to Vec, sort, and convert back
    let mut values: Vec<usize> = combined.to_vec();
    values.sort_by(|a, b| a.partial_cmp(b).unwrap());

    Array1::from_vec(values)
}

let true_pos_subset = select_random_elements(&labels_cpg, 4000, 1.0).unwrap();
let true_neg_subset = select_random_elements(&labels_cpg, 40000, 0.0).unwrap();

let all_indices = combine_and_sort(true_pos_subset, true_neg_subset);

let training_matrix_cpg = select_rows_by_indices(&full_data_matrix_cpg, &all_indices).unwrap();
let label_subset_cpg = labels_cpg.select(Axis(0), all_indices.as_slice().unwrap());

let training_dim=training_matrix_cpg.dim();
println!("Will train from matrix with dimensions {}x{}", training_dim.0, training_dim.1);

Will train from matrix with dimensions 44000x55


### 3. Train
We should now have everything we need to train our RF:

In [8]:
use biosphere::{RandomForest, RandomForestParameters, MaxFeatures};
use std::time::{Duration, Instant};

let random_forest_parameters = RandomForestParameters::default()
        .with_max_features(MaxFeatures::Value(2))
        .with_n_estimators(1000)
        .with_max_depth(None)
        .with_n_jobs(Some(8));

let mut start = Instant::now();
let mut model_cpg = RandomForest::new(random_forest_parameters);
model_cpg.fit(&training_matrix_cpg.view(), &label_subset_cpg.view());
let mut duration = start.elapsed();
eprintln!("Time taken for fitting model: {:?}", duration);


Time taken for fitting model: 17.088499875s


4.5s total (~40s total) as opposed to 2m 20s - ok, this is _much_ faster than SmartCore that I tried previously!

### 4. Benchmark
Let's benchmark by predicting all CpG positions on chr12:

In [9]:
start = Instant::now();
let predicted_labels = model_cpg.predict(&full_data_matrix_cpg.view());
duration = start.elapsed();
eprintln!("Time taken for predicting on all rows: {:?}", duration);

Time taken for predicting on all rows: 97.750351083s


~1 minute - not bad! This is substantially faster than smartcore, which took around 7 minutes with the "classifier" core, and a lot more with the regression core.

Let's now do some evaluation on the full dataset:

In [10]:
use smartcore::metrics::*;
let threshold: f64 = 0.75;

fn print_metrics(predicted_labels: &ArrayView1<f64>, observed_labels: &ArrayView1<f64>, threshold: f64) -> () {
    let all_pred_binary: Vec<f64> = predicted_labels.iter().map(|x| if *x>threshold {1.0} else {0.0} ).collect();
    let all_labels: Vec<f64> = observed_labels.to_vec().iter().map(|x| *x as f64).collect();
    let cpg_prec = precision(&all_pred_binary, &all_labels);
    let cpg_acc = accuracy(&all_pred_binary.iter().map(|x| *x as u32).collect::<Vec<u32>>(), &all_labels.iter().map(|x| *x as u32).collect());
    let cpg_f1 = f1(&all_pred_binary, &all_labels, 0.5);

    let cpg_recall = recall(&all_pred_binary, &all_labels);

    let (mut t_p, mut f_p, mut t_n, mut f_n) = (0, 0, 0, 0);
    for (i, p) in all_pred_binary.iter().enumerate() {
        let a = *p == 1.0;
        let b = observed_labels[i] == 1.0;

        let is_true = a == b;
        let is_pos = a == true;
        if is_true && is_pos {
            t_p = t_p + 1;
        } else if is_true && !is_pos {
            t_n = t_n + 1;
        } else if !is_true && is_pos {
            f_p = f_p + 1;
        } else {
            f_n = f_n + 1;
        }
    }
    println!("\t\tPredicted\nObserved\tSNP\tREF\n\tSNP\t{}\t{}\n\tREF\t{}\t{}\n", t_p, f_n, f_p, t_n);
    println!("Precision: {}\nRecall: {}\nAccuracy: {}\nF1: {}\n", cpg_prec, cpg_recall, cpg_acc, cpg_f1);
}

In [11]:
print_metrics(&predicted_labels.view(), &labels_cpg.view(), threshold);

		Predicted
Observed	SNP	REF
	SNP	22816	2575
	REF	2500	2179029

Precision: 0.9012482224680044
Recall: 0.8985861131897129
Accuracy: 0.99770041505809
F1: 0.9007145394970588



### 5. Serialise

In [13]:
use flate2::write::GzEncoder;
use std::io::prelude::*;
use flate2::Compression;
use bincode::serde::*;

// Save the model
fn write_model_to_file(model: &RandomForest, file_name: &str) -> () {
    let rf_cpg_bytes = bincode::serde::encode_to_vec(&model, bincode::config::standard()).expect("Cannot serialize the model");

    File::create(file_name)
        .and_then(|mut f| {
            let mut e = GzEncoder::new(f, Compression::default());
            e.write_all(&rf_cpg_bytes)
        })
        .expect("Can not persist model");
}

let mut file_name = "models/BS_RF_1000-2_CpG.rfz";
write_model_to_file(&model_cpg, &file_name);

## De-novo CpG positions

### 1. Load data

## 

In [14]:
let dn_cpg_data_file = "data/chr12_denovo_features.txt.gz";
let (full_data_matrix_dncpg, labels_dncpg) = tsv_to_matrix(dn_cpg_data_file, NewCpG).unwrap();

Found row with 57 columns


Finished reading 1124854 rows into a vector of size 61866970


### 2. Subsample

In [15]:

let true_pos_subset = select_random_elements(&labels_dncpg, 4000, 1.0).unwrap();
let true_neg_subset = select_random_elements(&labels_dncpg, 40000, 0.0).unwrap();

let all_indices = combine_and_sort(true_pos_subset, true_neg_subset);

let training_matrix_dncpg = select_rows_by_indices(&full_data_matrix_dncpg, &all_indices).unwrap();
let label_subset_dncpg = labels_dncpg.select(Axis(0), all_indices.as_slice().unwrap());

let training_dim=training_matrix_dncpg.dim();
println!("Will train from matrix with dimensions {}x{}", training_dim.0, training_dim.1);

Will train from matrix with dimensions 44000x55


### 3. Train

In [16]:
let random_forest_parameters = RandomForestParameters::default()
        .with_max_features(MaxFeatures::Value(2))
        .with_n_estimators(1000)
        .with_max_depth(None)
        .with_n_jobs(Some(8));

let mut start = Instant::now();
let mut model_dn = RandomForest::new(random_forest_parameters);
model_dn.fit(&training_matrix_dncpg.view(), &label_subset_dncpg.view());
let mut duration = start.elapsed();
eprintln!("Time taken for fitting model: {:?}", duration);

Time taken for fitting model: 15.601703s


### 4. Benchmark

In [17]:
start = Instant::now();
let predicted_labels_dn = model_dn.predict(&full_data_matrix_dncpg.view());
duration = start.elapsed();
eprintln!("Time taken for predicting on all rows: {:?}", duration);

Time taken for predicting on all rows: 44.427436167s


In [20]:
print_metrics(&predicted_labels_dn.view(), &labels_dncpg.view(), threshold);

		Predicted
Observed	SNP	REF
	SNP	21613	1792
	REF	1698	1099751

Precision: 0.9271588520440993
Recall: 0.9234351634266182
Accuracy: 0.9968973751260164
F1: 0.9264117137737999



Observed	SNP	REF
	SNP	21937	1468
	REF	2064	1099385

Precision: 0.914003583184034
Recall: 0.9372783593249305
Accuracy: 0.9968600369470171
F1: 0.9185656022577864



### 5. Serialise

In [21]:
let mut file_name = "models/BS_RF_1000-2_denovo.rfz";
write_model_to_file(&model_dn, &file_name);

## Other positions

### 1. Load data

In [22]:
let other_data_file = "data/chr12_other_features.txt.gz";
let (full_data_matrix_other, labels_other) = tsv_to_matrix(other_data_file, Other).unwrap();

Found row with 56 columns


Finished reading 8767912 rows into a vector of size 473467248


### 2. Subsample

In [23]:

let true_pos_subset = select_random_elements(&labels_other, 4000, 1.0).unwrap();
let true_neg_subset = select_random_elements(&labels_other, 40000, 0.0).unwrap();

let all_indices = combine_and_sort(true_pos_subset, true_neg_subset);

let training_matrix_other = select_rows_by_indices(&full_data_matrix_other, &all_indices).unwrap();
let label_subset_other = labels_other.select(Axis(0), all_indices.as_slice().unwrap());

let training_dim=training_matrix_other.dim();
println!("Will train from matrix with dimensions {}x{}", training_dim.0, training_dim.1);

Will train from matrix with dimensions 44000x54


### 3. Train

In [24]:
let random_forest_parameters = RandomForestParameters::default()
        .with_max_features(MaxFeatures::Value(2))
        .with_n_estimators(1000)
        .with_max_depth(None)
        .with_n_jobs(Some(8));

let mut start = Instant::now();
let mut model_other = RandomForest::new(random_forest_parameters);
model_other.fit(&training_matrix_other.view(), &label_subset_other.view());
let mut duration = start.elapsed();
eprintln!("Time taken for fitting model: {:?}", duration);

Time taken for fitting model: 14.707718584s


### 4. Benchmark

In [25]:
start = Instant::now();
let predicted_labels_other = model_other.predict(&full_data_matrix_other.view());
duration = start.elapsed();
eprintln!("Time taken for predicting on all rows: {:?}", duration);

Time taken for predicting on all rows: 321.481256209s


In [26]:
print_metrics(&predicted_labels_other.view(), &labels_other.view(), threshold);

		Predicted
Observed	SNP	REF
	SNP	104430	6248
	REF	10042	8647192

Precision: 0.9122754909497519
Recall: 0.9435479499087442
Accuracy: 0.9981420890173168
F1: 0.9183630396471122



### 5. Serialize

In [27]:
let mut file_name = "models/BS_RF_1000-2_other.rfz";
write_model_to_file(&model_other, &file_name);